The best open source for the identification of Tribes recognized by the U.S. government is the collection of Federal Register notices, published when some action has taken place (legislative or bureaucratic) to officially recognize a Tribe by a given name. Having items in Wikidata that represent and link to the FR notices is useful as a way to explicitly cite the specific notice where a Tribe is listed. Some of these were created in times past, but I found a good number of them missing, worked up the basic information to uniquely identify the missing references, and used the steps in this notebook to add them to Wikidata.

This covers the FR notices identifying "Indian Entities" back to 1995 where there is some type of digital representation available online and accessible via the [Federal Register Document Number](https://www.wikidata.org/wiki/Property:P1544) property, which uses a formatter URL to link to a landing page where the text of the notice can be accessed in a couple different ways. Prior to 1995, there are PDF files from scans that are often available, but those are not accessible in the same way through the online index/database. If we end up wanting to get those fully represented in the system, we'll need to make some tweaks to the Wikidata content model and use a different way of linking to the content. (See this [1979 item](https://www.wikidata.org/wiki/Q19068033) as an example we can work from.)

In [1]:
from wbmaker import WB
import os
import pandas as pd

In [17]:
wd = WB()

headers = {
    'User-Agent': os.environ.get('WB_BOT_USER_AGENT')
}

from datetime import datetime

def convert_date(date_string):
    """
    Convert a date string from YYYY-MM-DD format to a datetime object and a formatted string.
    
    Args:
        date_string: Date string in YYYY-MM-DD format (e.g., "1995-02-16")
    
    Returns:
        tuple: (datetime_object, formatted_string) where formatted_string is like "February 16, 1995"
    """
    dt_object = datetime.strptime(date_string, "%Y-%m-%d")
    formatted_string = dt_object.strftime("%B %d, %Y")
    return dt_object, formatted_string

new_fr_tribes = pd.read_csv('data/FR Tribe Lists.csv')

In [3]:
standard_title = "Indian Entities Recognized and Eligible To Receive Services from the United States Bureau of Indian Affairs"
standard_description = "official list of federally recognized Tribes in the United States"

standard_claims = {
    'P31': 'Q121840925',
    'P50': 'Q1010563',
    'P407': 'Q1860',
}

p_published_in = 'P1433'
q_published_in = 'Q5440362'

p_fr_id = 'P1544'

for _, row in new_fr_tribes.iterrows():
    dt, formatted_dt = convert_date(row['date'])
    title = f"{standard_title} ({formatted_dt})"
    description = f"{formatted_dt} {standard_description}"
    citation_parts = row['citation'].split(' ')
    fr_volume = citation_parts[0]
    fr_page = citation_parts[2]

    item = wd.wbi.item.new()
    item.labels.set('en', title)
    item.descriptions.set('en', description)
    item.aliases.set('en', row['citation'])

    for p, q in standard_claims.items():
        c = wd.datatypes.Item(prop_nr=p, value=q)
        item.claims.add(c)

    pub_qualifiers = []
    pub_qualifiers.append(wd.datatypes.String(prop_nr='P478', value=fr_volume))
    pub_qualifiers.append(wd.datatypes.String(prop_nr='P304', value=fr_page))
    pub_claim = wd.datatypes.Item(prop_nr=p_published_in, value=q_published_in, qualifiers=pub_qualifiers)
    item.claims.add(pub_claim)

    item.claims.add(wd.datatypes.String(prop_nr=p_fr_id, value=row['doc_number']))

    response = item.write(summary="Adding new FR notice")
    print(response.id, row['citation'])

Q137668715 61 FR 58211
Q137668716 62 FR 55270
Q137668717 63 FR 71941
Q137668718 65 FR 13298
Q137668719 67 FR 46328
Q137668720 68 FR 68180
Q137668721 70 FR 71194
Q137668722 72 FR 13648
Q137668723 73 FR 18553
Q137668724 74 FR 40218
Q137668725 75 FR 60810
Q137668726 75 FR 66124
Q137668727 77 FR 47868
Q137668728 78 FR 26384
Q137668729 79 FR 4748
Q137668730 80 FR 1942
Q137668731 81 FR 5019
Q137668732 81 FR 26826
Q137668733 82 FR 4915
Q137668734 83 FR 4235
Q137668735 86 FR 18552


# Check

The following SPARQL query can be used to pull all FR notices since we have type classified them the same way.

In [9]:
q_fr_notices = """
SELECT ?item ?itemLabel WHERE {
  ?item wdt:P31 wd:Q121840925 . # Item is instance of a Federal Register Notice
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }
}
"""

df_fr_notices = wd.sparql_query(q_fr_notices)

df_fr_notices

,item,itemLabel
0,http://www.wikidata.org/entity/Q19068033,Indian entities recognized and eligible to rec...
1,http://www.wikidata.org/entity/Q56884857,Indian Entities Recognized and Eligible To Rec...
2,http://www.wikidata.org/entity/Q121840943,Indian Entities Recognized by and Eligible To ...
3,http://www.wikidata.org/entity/Q127419548,Indian Entities Recognized by and Eligible To ...
4,http://www.wikidata.org/entity/Q137668125,Indian Entities Recognized and Eligible To Rec...
5,http://www.wikidata.org/entity/Q137668715,Indian Entities Recognized and Eligible To Rec...
6,http://www.wikidata.org/entity/Q137668716,Indian Entities Recognized and Eligible To Rec...
7,http://www.wikidata.org/entity/Q137668717,Indian Entities Recognized and Eligible To Rec...
8,http://www.wikidata.org/entity/Q137668718,Indian Entities Recognized and Eligible To Rec...
9,http://www.wikidata.org/entity/Q137668719,Indian Entities Recognized and Eligible To Rec...


# Oops

I initially forgot to add in the publication date claim, so this script runs through to get those from all titles and write/replace them on each item that the query returns.

In [18]:
from datetime import datetime

for _, row in df_fr_notices.iterrows():
    pub_date = row['itemLabel'].split('(')[-1].strip(')')
    dt_object = datetime.strptime(pub_date, "%B %d, %Y")
    wd_dt_object = wd.wb_dt(dt_object)

    pub_date_claim = wd.datatypes.Time(prop_nr='P577', time=wd_dt_object, precision=11)

    qid = row['item'].split('/')[-1]
    item = wd.wbi.item.get(qid)
    
    item.claims.add(pub_date_claim, action_if_exists=wd.wbi_enums.ActionIfExists.REPLACE_ALL)
    response = item.write(summary="Adding publication date claim")
    
    print(response.id)

Q19068033
Q56884857
Q121840943
Q127419548
Q137668125
Q137668715
Q137668716
Q137668717
Q137668718
Q137668719
Q137668720
Q137668721
Q137668722
Q137668723
Q137668724
Q137668725
Q137668726
Q137668727
Q137668728
Q137668729
Q137668730
Q137668731
Q137668732
Q137668733
Q137668734
Q137668735
